# AOI-PCB: Model Evaluation

This notebook evaluates a trained IC keypoint detection model on the validation dataset and visualises predicted vs actual corner keypoints overlaid on PCB images.

- **Red circles** — ground-truth corner positions
- **Blue circles** — model predictions

### Prerequisites
1. Install the package: `pip install -e ".[dev]"`
2. Generate the dataset: `python scripts/generate_dataset.py`
3. Train a model: `python scripts/train.py --output-dir experiments/run_YYYYMMDD_HHMMSS`

`MODEL_PATH` below defaults to the most recently created run. Override it to evaluate a specific run.

## 1. Setup

In [ ]:
import math
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf

from aoi_pcb.config_loader import Config
from aoi_pcb.data.encoder import DataEncoder
from aoi_pcb.data.utils import sort_alphanumeric
from aoi_pcb.model.loss import custom_loss
from aoi_pcb.model.metric import KeypointAlignmentMetric

# --- Configuration ---
# Defaults to the most recently modified run. Override to evaluate a specific run:
#   MODEL_PATH = "../experiments/run_YYYYMMDD_HHMMSS/model.keras"
_runs = sorted(
    Path("../experiments").glob("*/model.keras"),
    key=lambda p: p.stat().st_mtime,
    reverse=True,
)
MODEL_PATH = str(_runs[0]) if _runs else None

N_VISUALS = 20  # number of overlay images to display
N_COLS = 5  # images per row
cell_size = 5  # inches per cell

if MODEL_PATH is None:
    raise FileNotFoundError("No trained model found in ../experiments/. Run train.py first.")
print(f"Evaluating: {MODEL_PATH}")

# Load config from the run directory to match training conditions exactly
config = Config(str(Path(MODEL_PATH).parent / "config.json"))

## 2. Load Validation Data

We evaluate on the validation split — images the model has not been trained on — using the same `DataEncoder` as training.

In [ ]:
encoder = DataEncoder(config)

val = config.generator.val_data
data_dir = str(Path("..") / val.data_dir)
label_path = Path("..") / val.labels_dir / val.label_file

X, y, ref_coords, ref_center = encoder(data_dir, label_path)

print(f"Images : {X.shape}  dtype={X.dtype}")
print(f"Labels : {y.shape}  dtype={y.dtype}")

## 3. Load Model

The model is loaded with its custom loss and metric registered as `custom_objects` so Keras can deserialise them correctly.

In [ ]:
metric = KeypointAlignmentMetric(ref_center, ref_coords, config)

# compile=False skips Keras's attempt to deserialise the saved compile config,
# which fails for @tf.function-decorated losses. We re-compile immediately after.
model = tf.keras.models.load_model(MODEL_PATH, compile=False)
model.compile(loss=custom_loss, metrics=[metric.alignment_metric])
model.summary()

## 4. Evaluate

Reports the combined loss and alignment metric on the full validation set.

In [ ]:
results = model.evaluate(X, y, verbose=1)
print("\nEvaluation results:")
for name, value in zip(model.metrics_names, results):
    print(f"  {name}: {value:.6f}")

## 5. Predict

In [ ]:
predictions = model.predict(X)
print(f"Predictions shape: {predictions.shape}")

## 6. Visualise Predictions

Each image shows the PCB with predicted (blue) and ground-truth (red) IC corner keypoints overlaid. Close agreement between the two indicates good model performance.

Per the paper, the model correctly identifies the IC center even on images from PCB boards not seen during training.

In [ ]:
image_files = sort_alphanumeric(data_dir)
img_width = X.shape[1]
normalized = config.encoder.normalize_labels

n = min(N_VISUALS, len(image_files))
n_cols = min(N_COLS, n)
n_rows = math.ceil(n / n_cols)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * cell_size, n_rows * cell_size))
axes = np.array(axes).reshape(-1)  # flatten to 1-D regardless of grid shape

for idx in range(n):
    ax = axes[idx]
    img = X[idx].copy()

    actual = y[idx] * img_width if normalized else y[idx]
    predicted = predictions[idx] * img_width if normalized else predictions[idx]

    corners_actual = np.rint(actual.reshape(-1, 2)).astype("int32")
    corners_predicted = np.rint(predicted.reshape(-1, 2)).astype("int32")

    for corner in corners_actual:
        img = cv2.circle(img, tuple(corner), 4, (255, 0, 0), 2)
    for corner in corners_predicted:
        img = cv2.circle(img, tuple(corner), 4, (0, 0, 255), 2)

    ax.imshow(img)
    ax.set_title(f"Sample {idx}", fontsize=9)
    ax.axis("off")

# Hide any unused axes in the last row
for idx in range(n, len(axes)):
    axes[idx].set_visible(False)

plt.suptitle("Red = Ground Truth   |   Blue = Prediction", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 7. Per-sample Error Analysis

Compute the Euclidean distance between each predicted and actual corner for a quick per-sample breakdown.

In [ ]:
scale = img_width if normalized else 1

actual_px = y * scale
pred_px = predictions * scale

errors = np.linalg.norm(
    actual_px.reshape(-1, 4, 2) - pred_px.reshape(-1, 4, 2),
    axis=2,
)  # shape: (N, 4) — one distance per corner per sample

mean_error_per_corner = errors.mean(axis=0)
overall_mean = errors.mean()

print("Mean pixel error per corner:")
for i, e in enumerate(mean_error_per_corner):
    print(f"  Corner {i}: {e:.2f} px")
print(f"Overall mean error: {overall_mean:.2f} px")

plt.figure(figsize=(8, 4))
plt.hist(errors.flatten(), bins=40, color="steelblue", edgecolor="white")
plt.title("Distribution of Corner Prediction Errors")
plt.xlabel("Euclidean distance (pixels)")
plt.ylabel("Count")
plt.grid(True)
plt.tight_layout()
plt.show()